# 06 — Apprentissage supervisé (Volet A)

**Objectif** : à partir des features préparées par Asma (`features_supervised.csv`), prédire pour un couple (utilisateur, film) :
- **Classification** : l'utilisateur *aime* (note ≥ `LIKE_THRESHOLD`) ou *n'aime pas* le film → KNN, SVM, régression logistique.
- **Régression** : la note elle-même → régression linéaire, Ridge, arbre de décision.

**Protocole** (`docs/conventions.md`) :
- Les hyperparamètres sont choisis avec `GridSearchCV` **sur Validation**, jamais sur Test.
- La validation croisée (5 folds sur Train) mesure la stabilité des modèles retenus.
- Le Test est utilisé **une seule fois**, à la fin (`RUN_TEST`).

> **Statut** : en attente de `features_supervised.csv` (Asma). Ce notebook ne crée aucune feature : il les lit.
> Le format attendu (section 1) est à confirmer avec Asma.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import roc_curve
from sklearn.model_selection import GridSearchCV, KFold, PredefinedSplit, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.evaluation import classification_metrics, regression_metrics

PROCESSED = ROOT / "data" / "processed"

RANDOM_STATE = 42
LIKE_THRESHOLD = 4.0   # "aime" si note >= 4 (à aligner avec Asma)
RUN_TEST = False       # passer à True UNE SEULE FOIS, à la fin

# Style des graphiques
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "lines.linewidth": 2, "legend.frameon": False, "font.size": 10,
})

## 1. Données

Format **supposé** de `features_supervised.csv`, **à confirmer avec Asma** : une ligne par note, avec
- `userId`, `movieId` (identifiants, `int64`) ;
- `rating` : la note (cible) ;
- `split` : `train`, `validation` ou `test` ;
- toutes les autres colonnes numériques = features.

Si le format d'Asma est différent, seule la cellule ci-dessous change.

In [ ]:
FEATURES_FILE = PROCESSED / "features_supervised.csv"
ID_COLS = ["userId", "movieId"]
TARGET = "rating"
SPLIT_COL = "split"

if not FEATURES_FILE.exists():
    raise FileNotFoundError(f"{FEATURES_FILE} introuvable : fichier à produire par Asma (02_data_preprocessing).")

data = pd.read_csv(FEATURES_FILE)
candidates = data.drop(columns=ID_COLS + [TARGET, SPLIT_COL])
feature_cols = list(candidates.select_dtypes("number").columns)
ignored = sorted(set(candidates.columns) - set(feature_cols))
if ignored:
    print("Colonnes non numériques ignorées :", ignored)

parts = {s: data[data[SPLIT_COL] == s].reset_index(drop=True) for s in ("train", "validation", "test")}
X_train, X_val, X_test = (parts[s][feature_cols] for s in ("train", "validation", "test"))
y_train_reg, y_val_reg, y_test_reg = (parts[s][TARGET].to_numpy() for s in ("train", "validation", "test"))
y_train_cls, y_val_cls, y_test_cls = ((y >= LIKE_THRESHOLD).astype(int) for y in (y_train_reg, y_val_reg, y_test_reg))

print(f"{len(feature_cols)} features : {feature_cols}\n")
for name, s in [("Train", "train"), ("Validation", "validation"), ("Test", "test")]:
    print(f"{name:10s} {len(parts[s]):>6} lignes")
for name, y in [("Train", y_train_cls), ("Validation", y_val_cls)]:
    print(f"{name:10s} aime : {y.mean():.1%}   n'aime pas : {1 - y.mean():.1%}")

**Déséquilibre des classes** : on teste `class_weight="balanced"` dans la grille (régression logistique et SVM).
La recherche d'hyperparamètres garde l'option si elle améliore le F1 sur Validation. KNN n'a pas de `class_weight`.

### Recherche d'hyperparamètres sur Validation

`PredefinedSplit` dit à `GridSearchCV` : « entraîne sur Train, évalue sur Validation ». Chaque combinaison est donc
jugée sur Validation, et le Test n'est jamais vu. On réentraîne ensuite le meilleur modèle sur Train.

In [ ]:
X_tv = pd.concat([X_train, X_val], ignore_index=True)
val_split = PredefinedSplit(np.r_[np.full(len(X_train), -1), np.zeros(len(X_val), dtype=int)])


def tune(estimator, grid, y_train, y_val, scoring):
    # Choisit les hyperparamètres sur Validation, puis réentraîne sur Train
    if grid:
        search = GridSearchCV(estimator, grid, cv=val_split, scoring=scoring, refit=False, n_jobs=-1)
        search.fit(X_tv, np.r_[y_train, y_val])
        params = search.best_params_
    else:
        params = {}
    model = clone(estimator).set_params(**params).fit(X_train, y_train)
    return model, params


def short(params):
    return ", ".join(f"{k.split('__')[-1]}={v}" for k, v in params.items()) or "-"


def score_of(model, X):
    # Score de la classe 1 : probabilité si disponible, sinon distance à la frontière (SVM)
    return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)

## 2. Classification « aime / n'aime pas »

- **Régression logistique** et **SVM** : modèles linéaires, données standardisées.
- **SVM** : `LinearSVC` (noyau linéaire). Un SVM à noyau RBF (`SVC`) a un coût d'entraînement quadratique ou pire en nombre de lignes :
  sur ~80 000 notes, il est beaucoup trop lent. `LinearSVC` ne donne pas de probabilité : l'AUC est calculée sur `decision_function`.
- **KNN** : sensible à l'échelle, d'où la standardisation.
- **Critère de choix** : F1 sur Validation (équilibre précision / rappel sur la classe « aime »).

In [ ]:
classifiers = {
    "Régression logistique": (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        {"logisticregression__C": [0.01, 0.1, 1, 10],
         "logisticregression__class_weight": [None, "balanced"]}),
    "SVM linéaire": (
        make_pipeline(StandardScaler(), LinearSVC(random_state=RANDOM_STATE)),
        {"linearsvc__C": [0.01, 0.1, 1],
         "linearsvc__class_weight": [None, "balanced"]}),
    "KNN": (
        make_pipeline(StandardScaler(), KNeighborsClassifier()),
        {"kneighborsclassifier__n_neighbors": [15, 51, 101],
         "kneighborsclassifier__weights": ["uniform", "distance"]}),
}

cv_cls = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cls_models, cls_scores, cls_rows = {}, {}, []

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train_cls)
m = classification_metrics(y_val_cls, baseline.predict(X_val), score_of(baseline, X_val))
cls_rows.append({"Modèle": "Baseline (classe majoritaire)", "Hyperparamètres": "-",
                 **{k: m[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]}})

for name, (estimator, grid) in classifiers.items():
    t0 = time.time()
    model, params = tune(estimator, grid, y_train_cls, y_val_cls, scoring="f1")
    scores = score_of(model, X_val)
    m = classification_metrics(y_val_cls, model.predict(X_val), scores)
    cv = cross_validate(model, X_train, y_train_cls, cv=cv_cls, scoring=["f1", "roc_auc"], n_jobs=-1)
    cls_models[name], cls_scores[name] = model, scores
    cls_rows.append({"Modèle": name, "Hyperparamètres": short(params),
                     **{k: m[k] for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]},
                     "F1 CV (moy)": cv["test_f1"].mean(), "F1 CV (écart-type)": cv["test_f1"].std(),
                     "AUC CV (moy)": cv["test_roc_auc"].mean(), "Temps (s)": time.time() - t0})
    print(f"{name:22s} F1 val = {m['f1']:.4f} | AUC val = {m['roc_auc']:.4f} | {short(params)} | {time.time() - t0:.0f} s")

cls_table = pd.DataFrame(cls_rows).set_index("Modèle")

### Matrices de confusion et courbes ROC (Validation)

In [ ]:
labels = ["n'aime pas", "aime"]
fig, axes = plt.subplots(1, len(cls_models), figsize=(4.2 * len(cls_models), 3.8))
for ax, (name, model) in zip(axes, cls_models.items()):
    cm = classification_metrics(y_val_cls, model.predict(X_val))["confusion_matrix"]
    share = cm / cm.sum(axis=1, keepdims=True)  # couleur = % de la ligne réelle
    ax.imshow(share, cmap=BLUES, vmin=0, vmax=1)
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            color = "white" if share[i, j] > 0.5 else INK
            ax.text(j, i, f"{cm[i, j]}\n{share[i, j]:.0%}", ha="center", va="center", color=color)
    ax.set_xticks([0, 1], labels)
    ax.set_yticks([0, 1], labels)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")
    ax.set_title(name)
fig.suptitle("Matrices de confusion — Validation (% par ligne réelle)", color=INK)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color=MUTED, label="Hasard (AUC = 0.5)")
# SVM en pointillés : sa courbe peut se superposer à celle de la régression logistique
styles = ["-", (0, (4, 3)), "-"]
for color, style, (name, scores) in zip(SERIES, styles, cls_scores.items()):
    fpr, tpr, _ = roc_curve(y_val_cls, scores)
    ax.plot(fpr, tpr, color=color, linestyle=style,
            label=f"{name} (AUC = {cls_table.loc[name, 'roc_auc']:.3f})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal")
ax.set_xlabel("Taux de faux positifs")
ax.set_ylabel("Taux de vrais positifs (rappel)")
ax.set_title("Courbes ROC — Validation")
ax.legend(loc="lower right")
plt.show()

## 3. Régression de la note

- **Baseline** : moyenne globale de Train. Un modèle utile doit la battre.
- **Régression linéaire** (pas d'hyperparamètre), **Ridge** (pénalité `alpha`), **arbre de décision** (`max_depth`, `min_samples_leaf`).
- Les prédictions sont bornées à [0.5, 5], l'échelle des notes MovieLens.
- **Critère de choix** : RMSE sur Validation.

In [ ]:
regressors = {
    "Régression linéaire": (make_pipeline(StandardScaler(), LinearRegression()), {}),
    "Ridge": (make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": [0.1, 1, 10, 100, 1000]}),
    "Arbre de décision": (DecisionTreeRegressor(random_state=RANDOM_STATE),
                          {"max_depth": [4, 6, 8, 12], "min_samples_leaf": [20, 100]}),
}


def predict_rating(model, X):
    return np.clip(model.predict(X), 0.5, 5.0)


cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_models = {}
reg_rows = [{"Modèle": "Baseline (moyenne globale)", "Hyperparamètres": "-",
             **regression_metrics(y_val_reg, np.full(len(y_val_reg), y_train_reg.mean()))}]

for name, (estimator, grid) in regressors.items():
    t0 = time.time()
    model, params = tune(estimator, grid, y_train_reg, y_val_reg, scoring="neg_root_mean_squared_error")
    m = regression_metrics(y_val_reg, predict_rating(model, X_val))
    cv = cross_validate(model, X_train, y_train_reg, cv=cv_reg, scoring="neg_root_mean_squared_error", n_jobs=-1)
    reg_models[name] = model
    reg_rows.append({"Modèle": name, "Hyperparamètres": short(params), **m,
                     "RMSE CV (moy)": -cv["test_score"].mean(), "RMSE CV (écart-type)": cv["test_score"].std(),
                     "Temps (s)": time.time() - t0})
    print(f"{name:20s} RMSE val = {m['RMSE']:.4f} | MAE = {m['MAE']:.4f} | R² = {m['R2']:.4f} | {short(params)}")

reg_table = pd.DataFrame(reg_rows).set_index("Modèle")

## 4. Tableaux comparatifs (Validation)

In [ ]:
cls_table.sort_values("f1", ascending=False).round(4)

In [ ]:
reg_table.sort_values("RMSE").round(4)

### Choix du modèle final

Règles de décision, fixées **avant** de regarder le Test :
1. **Classification** : meilleur F1 sur Validation ; l'AUC départage. Le F1 en validation croisée doit être proche
   (écart-type faible), sinon le modèle est instable.
2. **Régression** : plus faible RMSE sur Validation, et le modèle doit battre la baseline.
3. À performance presque égale (écart < `TOLERANCE` = 0,005), on prend le plus simple, dans cet ordre :
   - classification : régression logistique → SVM linéaire → KNN. La régression logistique donne directement une
     probabilité entre 0 et 1, ce qu'attend le moteur hybride (`get_scores`). KNN est lent en prédiction : il compare
     chaque note à toutes les notes de Train ;
   - régression : régression linéaire → Ridge → arbre de décision.

> Justification finale à rédiger ici une fois les résultats obtenus avec `features_supervised.csv`.

In [ ]:
TOLERANCE = 0.005
SIMPLICITY = ["Régression logistique", "SVM linéaire", "KNN",
              "Régression linéaire", "Ridge", "Arbre de décision"]


def choose(table, names, metric, higher_is_better):
    # Parmi les modèles à moins de TOLERANCE du meilleur, prend le plus simple
    values = table.loc[names, metric]
    best = values.max() if higher_is_better else values.min()
    candidates = [n for n in names if abs(values[n] - best) < TOLERANCE]
    return min(candidates, key=SIMPLICITY.index)


best_cls = choose(cls_table, list(cls_models), "f1", higher_is_better=True)
best_reg = choose(reg_table, list(reg_models), "RMSE", higher_is_better=False)
print(f"Classification : {best_cls}  (F1 val = {cls_table.loc[best_cls, 'f1']:.4f}, AUC = {cls_table.loc[best_cls, 'roc_auc']:.4f})")
print(f"Régression     : {best_reg}  (RMSE val = {reg_table.loc[best_reg, 'RMSE']:.4f})")

## 5. Évaluation finale sur Test (une seule fois)

Ne lancer qu'une fois, avec les modèles choisis ci-dessus, en passant `RUN_TEST = True`.
Aucune décision ne doit être prise après avoir vu ces chiffres.

In [ ]:
if RUN_TEST:
    model = cls_models[best_cls]
    m = classification_metrics(y_test_cls, model.predict(X_test), score_of(model, X_test))
    print(f"Classification ({best_cls}) — Test")
    print({k: round(v, 4) for k, v in m.items() if k != "confusion_matrix"})
    print(m["confusion_matrix"])
    m = regression_metrics(y_test_reg, predict_rating(reg_models[best_reg], X_test))
    print(f"\nRégression ({best_reg}) — Test")
    print({k: round(v, 4) for k, v in m.items()})
else:
    print("Test non évalué (RUN_TEST = False).")